Puedes pedir las versiones impresa y ebook de *Think Python 3e* en
[Bookshop.org](https://bookshop.org/a/98697/9781098155438) y
[Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=oreilly20-20&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325).

In [1]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

Downloaded thinkpython.py
Downloaded diagram.py


Crédito: las fotos se descargaron de [Lorem Picsum](https://picsum.photos/), un servicio que proporciona imágenes de marcador de posición.
El nombre es una referencia a "lorem ipsum", que es un nombre para texto de marcador de posición.


In [2]:
# This cell downloads an archive file that contains the the files we'll
# use for the examples in this chapter.

download('https://github.com/AllenDowney/ThinkPython/raw/v3/photos.zip');

Downloaded photos.zip


In [ ]:
# WARNING: This cell removes the photos/ directory if it already exists.
# Any files already in the photos/ directory will be deleted.

# !rm -rf photos/

In [3]:
!unzip -o photos.zip

Archive:  photos.zip
   creating: photos/
  inflating: photos/notes.txt        
   creating: photos/mar-2023/
  inflating: photos/mar-2023/photo2.jpg  
  inflating: photos/mar-2023/photo1.jpg  
   creating: photos/jan-2023/
  inflating: photos/jan-2023/photo3.jpg  
  inflating: photos/jan-2023/photo2.jpg  
  inflating: photos/jan-2023/photo1.jpg  
   creating: photos/feb-2023/
  inflating: photos/feb-2023/photo2.jpg  
  inflating: photos/feb-2023/photo1.jpg  


# Archivos y bases de datos

La mayoría de los programas que hemos visto hasta ahora son **efímeros** en el sentido de que se ejecutan durante poco tiempo y producen output, pero cuando terminan, sus datos desaparecen.
Cada vez que ejecutas un programa efímero, empieza desde cero.

Otros programas son **persistentes**: se ejecutan durante mucho tiempo (o todo el tiempo); mantienen al menos parte de sus datos en almacenamiento a largo plazo; y, si se cierran y se reinician, continúan donde lo dejaron.

Una forma sencilla de que los programas mantengan sus datos es leer y escribir archivos de texto.
Una alternativa más versátil es almacenar datos en una base de datos.
Las bases de datos son archivos especializados que se pueden leer y escribir de forma más eficiente que los archivos de texto, y proporcionan capacidades adicionales.

En este capítulo escribiremos programas que leen y escriben archivos de texto y bases de datos, y como ejercicio escribirás un programa que busca duplicados en una colección de fotos.
Pero antes de poder trabajar con un archivo, tienes que encontrarlo, así que empezaremos con nombres de archivo, rutas y directorios.

## Nombres de archivo y rutas

Los archivos se organizan en **directorios**, también llamados "carpetas".
Todo programa en ejecución tiene un **directorio de trabajo actual**, que es el directorio por defecto para la mayoría de operaciones.
Por ejemplo, cuando abres un archivo, Python lo busca en el directorio de trabajo actual.

El módulo `os` proporciona funciones para trabajar con archivos y directorios ("os" significa "operating system").
Proporciona una función llamada `getcwd` que obtiene el nombre del directorio de trabajo actual.

In [4]:
# this cell replaces `os.cwd` with a function that returns a fake path

import os

def getcwd():
    return "/home/dinsdale"

os.getcwd = getcwd

In [5]:
import os

os.getcwd()

'/home/dinsdale'

El resultado en este ejemplo es el directorio personal de un usuario llamado `dinsdale`.
Una string como `'/home/dinsdale'` que identifica un archivo o directorio se llama **ruta**.

Un nombre de archivo sencillo como `'memo.txt'` también se considera una ruta, pero es un **ruta relativa** porque especifica un nombre de archivo relativo al directorio actual.
En este ejemplo, el directorio actual es `/home/dinsdale`, así que `'memo.txt'` equivale a la ruta completa `'/home/dinsdale/memo.txt'`.

Un ruta que empieza con `/` no depende del directorio actual -- se llama **ruta absoluta**.
Para encontrar la ruta absoluta de un archivo, puedes usar `abspath`.

In [6]:
os.path.abspath('memo.txt')

'/home/dinsdale/memo.txt'

El módulo `os` proporciona otras funciones para trabajar con nombres de archivo y rutas.
`listdir` devuelve una lista con el contenido del directorio indicado, incluyendo archivos y otros directorios.
Aquí tienes un ejemplo que lista el contenido de un directorio llamado `photos`.

In [7]:
os.listdir('photos')

['jan-2023', 'mar-2023', 'feb-2023', 'notes.txt']

Este directorio contiene un archivo de texto llamado `notes.txt` y tres directorios.
Los directorios contienen archivos de imagen en formato JPEG.

In [8]:
os.listdir('photos/jan-2023')

['photo2.jpg', 'photo3.jpg', 'photo1.jpg']

Para comprobar si existe un archivo o directorio, podemos usar `os.path.exists`.

In [9]:
os.path.exists('photos')

True

In [10]:
os.path.exists('photos/apr-2023')

False

Para comprobar si una ruta se refiere a un archivo o a un directorio, podemos usar `isdir`, que devuelve `True` si una ruta se refiere a un directorio.

In [11]:
os.path.isdir('photos')

True

Y `isfile`, que devuelve `True` si una ruta se refiere a un archivo.

In [12]:
os.path.isfile('photos/notes.txt')

True

Un desafío al trabajar con rutas es que se ven distintos en diferentes sistemas operativos.
En macOS y sistemas UNIX como Linux, los nombres de directorios y archivos en una ruta se separan con una barra inclinada, `/`.
Windows usa una barra invertida, `\`.
Así que, si ejecutas estos ejemplos en Windows, verás barras invertidas en los rutas y tendrás que reemplazar las barras inclinadas de los ejemplos.

O, para escribir código que funcione en ambos sistemas, puedes usar `os.path.join`, que une nombres de directorio y de archivo en una ruta usando una barra inclinada o invertida, según el sistema operativo que estés usando.

In [13]:
os.path.join('photos', 'jan-2023', 'photo1.jpg')

'photos/jan-2023/photo1.jpg'

Más adelante en este capítulo usaremos estas funciones para buscar en un conjunto de directorios y encontrar todos los archivos de imagen.

## f-strings

Una forma en que los programas almacenan datos es escribirlos en un archivo de texto.
Por ejemplo, supón que observas camellos y quieres registrar el número de camellos que has visto durante un periodo de observación.
Y supón que en un año y medio has visto `23` camellos.
Los datos de tu cuaderno de observación de camellos podrían verse así.

In [14]:
num_years = 1.5
num_camels = 23

Para escribir estos datos en un archivo, puedes usar el método `write`, que vimos en el Capítulo 8.
El argumento de `write` tiene que ser una string, así que si queremos poner otros valores en un archivo, tenemos que convertirlos a strings.
La forma más sencilla de hacerlo es con la función integrada `str`.

Así es como se ve:

In [15]:
writer = open('camel-spotting-book.txt', 'w')
writer.write(str(num_years))
writer.write(str(num_camels))
writer.close()

Eso funciona, pero `write` no añade un espacio ni una nueva línea a menos que lo incluyas explícitamente.
Si volvemos a leer el archivo, vemos que los dos números quedan pegados.

In [16]:
open('camel-spotting-book.txt').read()

'1.523'

Como mínimo, deberíamos añadir espacios en blanco entre los números.
Y ya que estamos, añadamos algo de texto explicativo.

Para escribir una combinación de strings y otros valores, podemos usar una **f-string**, que es una string que tiene la letra `f` antes de la comilla de apertura y contiene una o más expresiones de Python entre llaves.
La siguiente f-string contiene una expresión, que es el nombre de una variable.

In [17]:
f'I have spotted {num_camels} camels'

'I have spotted 23 camels'

El resultado es una string donde la expresión se ha evaluado y se ha reemplazado por el resultado.
Puede haber más de una expresión.

In [18]:
f'In {num_years} years I have spotted {num_camels} camels'

'In 1.5 years I have spotted 23 camels'

Y las expresiones pueden contener operadores y llamadas a función.

In [19]:
line = f'In {round(num_years * 12)} months I have spotted {num_camels} camels'
line

'In 18 months I have spotted 23 camels'

Así que podríamos escribir los datos en un archivo de texto así.

In [20]:
writer = open('camel-spotting-book.txt', 'w')
writer.write(f'Years of observation: {num_years}\n')
writer.write(f'Camels spotted: {num_camels}\n')
writer.close()

Ambas f-strings terminan con la secuencia `\n`, que añade un carácter de nueva línea.

Podemos volver a leer el archivo así:

In [21]:
data = open('camel-spotting-book.txt').read()
print(data)

Years of observation: 1.5
Camels spotted: 23



En una f-string, una expresión entre llaves se convierte en una string, así que puedes incluir listas, diccionarios y otros tipos.

In [22]:
t = [1, 2, 3]
d = {'one': 1}
f'Here is a list {t} and a dictionary {d}'

"Here is a list [1, 2, 3] and a dictionary {'one': 1}"

Si una f-string contiene una expresión no válida, el resultado es un error.

In [23]:
%%expect TypeError

f'This is not a valid expression {t + 2}'

TypeError: can only concatenate list (not "int") to list

## YAML

Una de las razones por las que los programas leen y escriben archivos es para almacenar **datos de configuración**, que son información que especifica qué debe hacer el programa y cómo.

Por ejemplo, en un programa que busca fotos duplicadas, podríamos tener un diccionario llamado `config` que contiene el nombre del directorio donde buscar, el nombre de otro directorio donde debería almacenar los resultados y una lista de extensiones de archivo que debería usar para identificar archivos de imagen.

Así podría verse:

In [24]:
config = {
    'photo_dir': 'photos',
    'data_dir': 'photo_info',
    'extensions': ['jpg', 'jpeg'],
}

Para escribir estos datos en un archivo de texto, podríamos usar f-strings, como en la sección anterior. Pero es más fácil usar un módulo llamado `yaml` que está diseñado justo para este tipo de cosas.

El módulo `yaml` proporciona funciones para trabajar con archivos YAML, que son archivos de texto con un formato pensado para que sean fáciles de leer y escribir tanto para humanos *como* para programas.

Aquí tienes un ejemplo que usa la función `dump` para escribir el diccionario `config` en un archivo YAML.

In [25]:
# this cell installs the pyyaml package, which provides the yaml module

try:
    import yaml
except ImportError:
    !pip install pyyaml

In [26]:
import yaml

config_filename = 'config.yaml'
writer = open(config_filename, 'w')
yaml.dump(config, writer)
writer.close()

Si volvemos a leer el contenido del archivo, podemos ver cómo es el formato YAML.

In [27]:
readback = open(config_filename).read()
print(readback)

data_dir: photo_info
extensions:
- jpg
- jpeg
photo_dir: photos



Ahora podemos usar `safe_load` para volver a leer el archivo YAML.

In [28]:
reader = open(config_filename)
config_readback = yaml.safe_load(reader)
config_readback

{'data_dir': 'photo_info',
 'extensions': ['jpg', 'jpeg'],
 'photo_dir': 'photos'}

El resultado es un nuevo diccionario que contiene la misma información que el original, pero no es el mismo diccionario.

In [29]:
config is config_readback

False

Convertir un objeto como un diccionario en una string se llama **serialización**.
Convertir la string de vuelta en un objeto se llama **deserialización**.
Si serializas y luego deserializas un objeto, el resultado debería ser equivalente al original.

## Shelve

Hasta ahora hemos estado leyendo y escribiendo archivos de texto -- ahora consideremos las bases de datos.
Una **base de datos** es un archivo organizado para almacenar datos.
Algunas bases de datos se organizan como una tabla con filas y columnas de información.
Otras se organizan como un diccionario que mapea claves a valores; a veces se llaman **almacenes clave-valor**.

El módulo `shelve` proporciona funciones para crear y actualizar un almacén clave-valor llamado "shelf".
Como ejemplo, crearemos un shelf para contener captions de las figuras del directorio `photos`.
Usaremos el diccionario `config` para obtener el nombre del directorio donde deberíamos poner el shelf.

In [30]:
config['data_dir']

'photo_info'

Podemos usar `os.makedirs` para crear este directorio, si todavía no existe.

In [31]:
os.makedirs(config['data_dir'], exist_ok=True)

Y `os.path.join` para crear una ruta que incluya el nombre del directorio y el nombre del archivo del shelf, `captions`.

In [32]:
db_file = os.path.join(config['data_dir'], 'captions')
db_file

'photo_info/captions'

Ahora podemos usar `shelve.open` para abrir el archivo del shelf.
El argumento `c` indica que el archivo debería crearse si es necesario.

In [33]:
import shelve

db = shelve.open(db_file, 'c')
db

Si obtienes un error como `db type could not be determined`, la causa más probable es que ya exista un archivo con el mismo nombre, pero que no sea una base de datos shelve válida (por ejemplo, puede estar corrupto o haber sido creado por otra cosa).

En ese caso, la solución más sencilla es eliminar el archivo existente y volver a ejecutar el código para que `shelve.open` pueda crear una base de datos nueva.

El valor de retorno es oficialmente un objeto `DbfilenameShelf`, llamado de manera más informal un objeto shelf.

El objeto shelf se comporta de muchas maneras como un diccionario.
Por ejemplo, podemos usar el operador de corchetes para añadir un elemento, que es un mapeo de una clave a un valor.

In [34]:
key = 'jan-2023/photo1.jpg'
db[key] = 'Cat nose'

En este ejemplo, la clave es la ruta a un archivo de imagen y el valor es una string que describe la imagen.

También usamos el operador de corchetes para buscar una clave y obtener el valor correspondiente.

In [35]:
value = db[key]
value

'Cat nose'

Si haces otra asignación a una clave existente, `shelve` reemplaza el valor anterior.

In [36]:
db[key] = 'Close up view of a cat nose'
db[key]

'Close up view of a cat nose'

Algunos métodos de diccionario, como `keys`, `values` e `items`, también funcionan con objetos shelf.

In [37]:
list(db.keys())

['jan-2023/photo1.jpg']

In [38]:
list(db.values())

['Close up view of a cat nose']

Podemos usar el operador `in` para comprobar si una clave aparece en el shelf.

In [39]:
key in db

True

Y podemos usar una sentencia `for` para iterar sobre las claves.

In [40]:
for key in db:
    print(key, ':', db[key])

jan-2023/photo1.jpg : Close up view of a cat nose


Como con otros archivos, deberías cerrar la base de datos cuando termines.

In [41]:
db.close()

Ahora, si listamos el contenido del directorio de datos, vemos dos archivos.

In [42]:
# When you open a shelve file, a backup file is created that has the suffix `.bak`.
# If you run this notebook more than once, you might see that file left behind.
# This cell removes it so the output shown in the book is correct.

!rm -f photo_info/captions.bak

In [43]:
os.listdir(config['data_dir'])

['captions.db']

`captions.dat` contiene los datos que acabamos de almacenar.
`captions.dir` contiene información sobre la organización de la base de datos que hace que el acceso sea más eficiente.
El sufijo `dir` significa "directorio", pero no tiene nada que ver con los directorios con los que hemos estado trabajando y que contienen archivos.

## Almacenar estructuras de datos

En el ejemplo anterior, las claves y valores del shelf son strings.
Pero también podemos usar un shelf para contener estructuras de datos como listas y diccionarios.

Como ejemplo, volvamos al ejemplo de anagramas de un ejercicio en el [Capítulo 11](section_exercise_11).
Recuerda que hicimos un diccionario que mapea una string ordenada de letras a la
lista de palabras que se pueden formar con esas letras.
Por ejemplo, la clave `'opst'` mapea a la lista `['opts', 'post', 'pots', 'spot', 'stop', 'tops']`.

Usaremos la siguiente función para ordenar las letras de una palabra.

In [44]:
def sort_word(word):
    return ''.join(sorted(word))

Y aquí tienes un ejemplo.

In [45]:
word = 'pots'
key = sort_word(word)
key

'opst'

Ahora abramos un shelf llamado `anagram_map`.
El argumento `'n'` significa que siempre deberíamos crear un shelf nuevo y vacío, aunque ya exista uno.

In [46]:
db = shelve.open('anagram_map', 'n')

Ahora podemos añadir un elemento al shelf así.

In [47]:
db[key] = [word]
db[key]

['pots']

En este elemento, la clave es una string y el valor es una lista de strings.

Ahora supón que encontramos otra palabra que contiene las mismas letras, como `tops`

In [48]:
word = 'tops'
key = sort_word(word)
key

'opst'

La clave es la misma que en el ejemplo anterior, así que queremos añadir una segunda palabra a la misma lista de strings.
Así es como lo haríamos si `db` fuera un diccionario.

In [49]:
db[key].append(word)          # INCORRECT

Pero si lo ejecutamos y luego buscamos la clave en el shelf, parece que no se ha actualizado.

In [50]:
db[key]

['pots']

Este es el problema: cuando buscamos la clave, obtenemos una lista de strings, pero si modificamos la lista de strings, eso no afecta al shelf.
Si queremos actualizar el shelf, tenemos que leer el valor antiguo, actualizarlo y luego escribir el nuevo valor de vuelta en el shelf.

In [ ]:
anagram_list = db[key]
anagram_list.append(word)
db[key] = anagram_list

Ahora el valor del shelf está actualizado.

In [51]:
db[key]

['pots']

Como ejercicio, puedes terminar este ejemplo leyendo la lista de palabras y almacenando todos los anagramas en un shelf.

In [52]:
db.close()

## Comprobar archivos equivalentes

Ahora volvamos al objetivo de este capítulo: buscar archivos diferentes que contienen los mismos datos.
Una forma de comprobarlo es leer el contenido de ambos archivos y compararlo.

Si los archivos contienen imágenes, tenemos que abrirlos con el modo `'rb'`, donde `'r'` significa que queremos leer el contenido y `'b'` indica **modo binario**.
En modo binario, el contenido no se interpreta como texto -- se trata como una secuencia de bytes.

Aquí tienes un ejemplo que abre y lee un archivo de imagen.

In [53]:
path1 = 'photos/jan-2023/photo1.jpg'
data1 = open(path1, 'rb').read()
type(data1)

bytes

El resultado de `read` es un objeto `bytes` -- como sugiere el nombre, contiene una secuencia de bytes.

En general, el contenido de un archivo de imagen no es legible para humanos.
Pero si leemos el contenido de un segundo archivo, podemos usar el operador `==` para comparar.

In [54]:
path2 = 'photos/jan-2023/photo2.jpg'
data2 = open(path2, 'rb').read()
data1 == data2

False

Estos dos archivos no son equivalentes.

Encapsulemos lo que tenemos hasta ahora en una función.

In [55]:
def same_contents(path1, path2):
    data1 = open(path1, 'rb').read()
    data2 = open(path2, 'rb').read()
    return data1 == data2

Si solo tenemos dos archivos, esta función es una buena opción.
Pero supón que tenemos un gran número de archivos y queremos saber si dos cualesquiera contienen los mismos datos.
Sería ineficiente comparar cada par de archivos.

Una alternativa es usar una **función hash**, que toma el contenido de un archivo y calcula un **digest**, que normalmente es un entero grande.
Si dos archivos contienen los mismos datos, tendrán el mismo digest.
Si dos archivos son diferentes, *casi siempre* tendrán digests diferentes.

El módulo `hashlib` proporciona varias hash funciones -- la que usaremos se llama `md5`.
Empezaremos usando `hashlib.md5` para crear un objeto `HASH`.

In [56]:
import hashlib

md5_hash = hashlib.md5()
type(md5_hash)

_hashlib.HASH

El objeto `HASH` proporciona un método `update` que toma el contenido del archivo como argumento.

In [57]:
md5_hash.update(data1)

Ahora podemos usar `hexdigest` para obtener el digest como una string de dígitos hexadecimales que representan un entero en base 16.

In [58]:
digest = md5_hash.hexdigest()
digest

'aa1d2fc25b7ae247b2931f5a0882fa37'

La siguiente función encapsula estos pasos.

In [59]:
def md5_digest(filename):
    data = open(filename, 'rb').read()
    md5_hash = hashlib.md5()
    md5_hash.update(data)
    digest = md5_hash.hexdigest()
    return digest

Si hacemos hash del contenido de un archivo diferente, podemos confirmar que obtenemos un digest distinto.

In [60]:
filename2 = 'photos/feb-2023/photo2.jpg'
md5_digest(filename2)

'6a501b11b01f89af9c3f6591d7f02c49'

Ahora tenemos casi todo lo que necesitamos para encontrar archivos equivalentes.
El último paso es buscar en un directorio y encontrar todos los archivos de imagen.

## Recorrer directorios

La siguiente función toma como argumento el directorio donde queremos buscar.
Usa `listdir` para iterar sobre el contenido del directorio.
Cuando encuentra un archivo, imprime su ruta completa.
Cuando encuentra un directorio, se llama a sí misma recursivamente para buscar en el subdirectorio.

In [61]:
def walk(dirname):
    for name in os.listdir(dirname):
        path = os.path.join(dirname, name)

        if os.path.isfile(path):
            print(path)
        elif os.path.isdir(path):
            walk(path)

Podemos usarla así:

In [62]:
walk('photos')

photos/jan-2023/photo2.jpg
photos/jan-2023/photo3.jpg
photos/jan-2023/photo1.jpg
photos/mar-2023/photo2.jpg
photos/mar-2023/photo1.jpg
photos/feb-2023/photo2.jpg
photos/feb-2023/photo1.jpg
photos/notes.txt


El orden de los resultados depende de detalles del sistema operativo.

## Depuración

Cuando lees y escribes archivos, puedes encontrarte con problemas relacionados con espacios en blanco.
Estos errores pueden ser difíciles de depurar porque los caracteres de espacio en blanco normalmente son invisibles.
Por ejemplo, aquí tienes una string que contiene espacios, un tab representado por la secuencia `\t` y una nueva línea representada por la secuencia `\n`.
Cuando la imprimimos, no vemos los caracteres de espacio en blanco.

In [63]:
s = '1 2\t 3\n 4'
print(s)

1 2	 3
 4


La función integrada `repr` puede ayudar. Toma cualquier objeto como argumento y devuelve una representación en string del objeto.
Para strings, representa los caracteres de espacio en blanco con secuencias de barra invertida.

In [64]:
print(repr(s))

'1 2\t 3\n 4'


Esto puede ser útil para depurar.

Otro problema que puedes encontrar es que distintos sistemas usan distintos caracteres para indicar el final de una línea. Algunos sistemas usan una nueva línea, representada como `\n`. Otros usan un carácter de retorno, representado como `\r`.
Algunos usan ambos. Si mueves archivos entre sistemas diferentes, estas
inconsistencias pueden causar problemas.

La capitalización de los nombres de archivo es otro problema que puedes encontrar si trabajas con distintos sistemas operativos.
En macOS y UNIX, los nombres de archivo pueden contener letras minúsculas y mayúsculas, dígitos y la mayoría de símbolos.
Pero muchas aplicaciones de Windows ignoran la diferencia entre letras minúsculas y mayúsculas, y varios símbolos que se permiten en macOS y UNIX no se permiten en Windows.

## Glosario

**efímero:**
Un programa efímero normalmente se ejecuta durante poco tiempo y, cuando termina, sus datos se pierden.

**persistente:**
 Un programa persistente se ejecuta indefinidamente y mantiene al menos parte de sus datos en almacenamiento permanente.

**directorio:**
Una colección de archivos y otros directorios.

**directorio de trabajo actual:**
El directorio por defecto usado por un programa a menos que se especifique otro directorio.

**ruta:**
 Una string que especifica una secuencia de directorios, que a menudo conduce a un archivo.

**ruta relativa:**
Un ruta que empieza desde el directorio de trabajo actual, o desde algún otro directorio especificado.

**ruta absoluta:**
Un ruta que no depende del directorio actual.

**f-string:**
Una string que tiene la letra `f` antes de la comilla de apertura y contiene una o más expresiones entre llaves.

**datos de configuración:**
Datos, a menudo almacenados en un archivo, que especifican qué debe hacer un programa y cómo.

**serialización:**
Convertir un objeto en una string.

**deserialización:**
Convertir una string en un objeto.

**base de datos:**
 Un archivo cuyo contenido está organizado para realizar ciertas operaciones de manera eficiente.

**almacenes clave-valor:**
Una base de datos cuyo contenido está organizado como un diccionario con claves que corresponden a valores.

**modo binario:**
Una forma de abrir un archivo para que el contenido se interprete como una secuencia de bytes en lugar de como una secuencia de caracteres.

**función hash:**
Una función que toma un objeto y calcula un entero, que a veces se llama digest.

**digest:**
El resultado de una función hash, especialmente cuando se usa para comprobar si dos objetos son iguales.

## Ejercicios

In [65]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

Exception reporting mode: Verbose


### Pregunta a un asistente virtual

Hay varios temas que surgieron en este capítulo que no expliqué en detalle.
Aquí tienes algunas preguntas que puedes hacerle a un asistente virtual para obtener más información.

* "¿Cuáles son las diferencias entre programas efímeros y persistentes?"

* "¿Cuáles son algunos ejemplos de programas persistentes?"

* "¿Cuál es la diferencia entre una ruta relativa y una ruta absoluta?"

* "¿Por qué el módulo `yaml` tiene funciones llamadas `load` y `safe_load`?"

* "Cuando escribo un shelf de Python, ¿qué son los archivos con sufijos `dat` y `dir`?"

* "Además de los almacenes clave-valor, ¿qué otros tipos de bases de datos existen?"

* "Cuando leo un archivo, ¿cuál es la diferencia entre modo binario y modo texto?"

* "¿Cuáles son las diferencias entre un objeto bytes y una string?"

* "¿Qué es una función hash?"

* "¿Qué es un digest MD5?"

Como siempre, si te quedas atascado en alguno de los ejercicios siguientes, considera pedir ayuda a un AV.  Junto con tu pregunta, quizá quieras pegar las funciones relevantes de este capítulo.

### Ejercicio

Escribe una función llamada `replace_all` que tome como argumentos una string patrón, una string de reemplazo y dos nombres de archivo.
Debería leer el primer archivo y escribir el contenido en el segundo archivo (creándolo si es necesario).
Si la string patrón aparece en cualquier parte del contenido, debería reemplazarse por la string de reemplazo.

Aquí tienes un esquema de la función para empezar.

In [66]:
def replace_all(old, new, source_path, dest_path):
    # read the contents of the source file
    reader = open(source_path)

    # replace the old string with the new
    content = reader.read()

    # write the result into the destination file
    new_content = content.replace(old, new)
    writer_content = open(dest_path, 'w')
    writer_content.write(new_content)
    reader.close()
    writer.close()


In [ ]:
# Solution goes here

Para probar tu función, lee el archivo `photos/notes.txt`, reemplaza `'photos'` por `'images'` y escribe el resultado en el archivo `photos/new_notes.txt`.

In [67]:
source_path = 'photos/notes.txt'
open(source_path).read()

'These photos are from Lorem Picsum at https://picsum.photos\n'

In [68]:
dest_path = 'photos/new_notes.txt'
old = 'photos'
new = 'images'
replace_all(old, new, source_path, dest_path)

In [69]:
open(dest_path).read()

'These images are from Lorem Picsum at https://picsum.images\n'

### Ejercicio

En [una sección anterior](section_storing_data_structure), usamos el módulo `shelve` para crear un almacén clave-valor que mapea una string ordenada de letras a una lista de anagramas.
Para terminar el ejemplo, escribe una función llamada `add_word` que tome como argumentos una string y un objeto shelf.

Debería ordenar las letras de la palabra para crear una clave, y luego comprobar si la clave ya está en el shelf. Si no, debería crear una lista que contenga la nueva palabra y añadirla al shelf. Si sí, debería añadir la nueva palabra al valor existente.

In [73]:
# Solution goes here
# Funcion para ordenar las letras de word
def sort_word(word):
    return ''.join(sorted(word))

def add_word(word, db):
    key = sort_word(word)
    if key not in db:
       anagram_list = [word]   # crear lista con la palabra
       db[key] = anagram_list  # guardar lista en db usando key
    else:
       anagram_list = db[key]  # obtener la lista existente
       anagram_list.append(word)  # agregar la palabra a la lista
       db[key] = anagram_list   # guardar la lista nuevamente en db


In [80]:
import shelve

db = shelve.open('anagram_map', 'n')

In [81]:
add_word('pots', db)
add_word('tops', db)
add_word('stop', db)
add_word('spot', db)

In [82]:
print(db['opst'])

['pots', 'tops', 'stop', 'spot']


In [83]:
db.close()

Puedes usar este bucle para probar tu función.

In [84]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

In [85]:
word_list = open('words.txt').read().split()

db = shelve.open('anagram_map', 'n')
for word in word_list:
    add_word(word, db)

Si todo funciona, deberías poder buscar una clave como `'opst'` y obtener una lista de palabras que se pueden formar con esas letras.

In [86]:
db['opst']

['opts', 'post', 'pots', 'spot', 'stop', 'tops']

In [87]:
for key, value in db.items():
    if len(value) > 8:
        print(value)

['estrin', 'inerts', 'insert', 'inters', 'niters', 'nitres', 'sinter', 'triens', 'trines']
['apers', 'asper', 'pares', 'parse', 'pears', 'prase', 'presa', 'rapes', 'reaps', 'spare', 'spear']
['alerts', 'alters', 'artels', 'estral', 'laster', 'ratels', 'salter', 'slater', 'staler', 'stelar', 'talers']
['least', 'setal', 'slate', 'stale', 'steal', 'stela', 'taels', 'tales', 'teals', 'tesla']
['capers', 'crapes', 'escarp', 'pacers', 'parsec', 'recaps', 'scrape', 'secpar', 'spacer']


In [79]:
db.close()

### Ejercicio

En una colección grande de archivos, puede haber más de una copia del mismo archivo, almacenada en distintos directorios o con distintos nombres de archivo.
El objetivo de este ejercicio es buscar duplicados.
Como ejemplo, trabajaremos con archivos de imagen en el directorio `photos`.

Así es como funcionará:

* Usaremos la función `walk` de [](section_walking_directories) para buscar en este directorio archivos que terminen con una de las extensiones en `config['extensions']`.

* Para cada archivo, usaremos `md5_digest` de [](section_md5_digest) para calcular un digest del contenido.

* Usando un shelf, haremos un mapeo de cada digest a una lista de rutas con ese digest.

* Finalmente, buscaremos en el shelf cualquier digest que mapee a múltiples archivos.

* Si encontramos alguno, usaremos `same_contents` para confirmar que los archivos contienen los mismos datos.

Voy a sugerir algunas funciones para escribir primero, y luego lo juntaremos todo.

1. Para identificar archivos de imagen, escribe una función llamada `is_image` que tome una ruta y una lista de extensiones de archivo, y devuelva `True` si la ruta termina con una de las extensiones de la lista. Pista: usa `os.path.splitext` -- o pide a un asistente virtual que escriba esta función por ti.

In [89]:
# Solution goes here
def is_image (path, extensions):
    nombre, extension = os.path.splitext(path)
    extension = extension[1:]

    if extension in extensions:
       return True
    else:
       return False

Puedes usar `doctest` para probar tu función.

In [90]:
from doctest import run_docstring_examples

def run_doctests(func):
    run_docstring_examples(func, globals(), name=func.__name__)

run_doctests(is_image)

2. Escribe una función llamada `add_path` que tome como argumentos una ruta y un shelf. Debería usar `md5_digest` para calcular un digest del contenido del archivo. Luego debería actualizar el shelf, ya sea creando un nuevo elemento que mapee el digest a una lista que contenga la ruta, o añadiendo la ruta a la lista si ya existe.

In [91]:
# Solution goes here
def add_path (path, db):
    digest = md5_digest(path)
    if digest not in db:
       path_list = [path]   # crear lista con la ruta
       db[digest] = path_list  # guardar lista en db usando digest
    else:
       path_list = db[digest]  # obtener la lista existente
       path_list.append(path)  # agregar la ruta a la lista
       db[digest] = path_list   # guardar la lista nuevamente en db


3. Escribe una versión de `walk` llamada `walk_images` que tome un directorio y recorra los archivos del directorio y sus subdirectorios. Para cada archivo, debería usar `is_image` para comprobar si es un archivo de imagen y `add_path` para añadirlo al shelf.

In [93]:
# Solution goes here
def walk_images (dirname):
    for name in os.listdir(dirname):
        path = os.path.join(dirname, name)   # construir path
        if os.path.isfile(path):    # si es archivo
            if is_image(path, config['extensions']):   # si es imagen
                add_path(path, db)
        elif os.path.isdir(path):   # si es directorio
            walk_images(path)  # llamar walk_images(recursivamente)

Cuando todo funcione, puedes usar el siguiente programa para crear el shelf, buscar en el directorio `photos` y añadir rutas al shelf, y luego comprobar si hay múltiples archivos con el mismo digest.

In [94]:
db = shelve.open('photos/digests', 'n')
walk_images('photos')

for digest, paths in db.items():
    if len(paths) > 1:
        print(paths)

['photos/jan-2023/photo1.jpg', 'photos/mar-2023/photo2.jpg']


Deberías encontrar un par de archivos que tienen el mismo digest.
Usa `same_contents` para comprobar si contienen los mismos datos.

In [96]:
# Solution goes here
same_contents(
    'photos/jan-2023/photo1.jpg',
    'photos/mar-2023/photo2.jpg'
)

True

In [99]:
len(paths)

1

[Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Código license: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)

Traducción al español por midudev (Miguel Ángel Durán).